# **Fine-tune the Qwen 0.5B Instruct base model**

In [5]:
from importlib.metadata import PackageNotFoundError, version

packages = [
    "transformers",
    "datasets",
    "peft",
    "trl",
    "bitsandbytes",
    "accelerate",
    "tokenizers",
    "sentencepiece",
    "safetensors",
    "scikit-learn",
    "pandas",
    "matplotlib",
]

for package in packages:
    try:
        print(f"{package:20s}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package:20s}: NOT INSTALLED")

transformers        : 5.20.2
datasets            : 4.8.5
peft                : 0.21.2
trl                 : NOT INSTALLED
bitsandbytes        : 0.50.2
accelerate          : 1.15.0
tokenizers          : 0.23.3
sentencepiece       : 0.2.2
safetensors         : 0.8.0
scikit-learn        : 1.6.1
pandas              : 2.2.3
matplotlib          : 3.10.0


In [6]:
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv
Saving validation.csv to validation.csv


In [9]:
import pandas as pd

train_df = pd.read_csv("/content/data/processed/train.csv")
val_df = pd.read_csv("/content/data/processed/validation.csv")

print("TRAIN")
print("Shape:", train_df.shape)
print("Columns:", train_df.columns.tolist())

print("\nVALIDATION")
print("Shape:", val_df.shape)
print("Columns:", val_df.columns.tolist())

assert len(train_df) == 19419
assert len(val_df) == 2427

required_columns = {"instruction", "category", "intent"}

assert required_columns.issubset(train_df.columns)
assert required_columns.issubset(val_df.columns)

print("\nDataset verification successful.")

TRAIN
Shape: (19419, 6)
Columns: ['flags', 'instruction', 'category', 'intent', 'response', 'normalized_instruction']

VALIDATION
Shape: (2427, 6)
Columns: ['flags', 'instruction', 'category', 'intent', 'response', 'normalized_instruction']

Dataset verification successful.


**Load the Qwen tokenizer and measure sequence lengths**

In [11]:
import json
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer

#load the model tokenizer - Qwen tokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

print("Tokenizer:", tokenizer.__class__.__name__)
print("EOS token:", tokenizer.eos_token)
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token:", tokenizer.pad_token)
print("Chat template available:", tokenizer.chat_template is not None)


#system prompt

SYSTEM_PROMPT = (
    "Classify customer support requests. "
    'Return only a JSON object with keys "category" and "intent".'
)

Tokenizer: Qwen2Tokenizer
EOS token: <|im_end|>
EOS token ID: 151645
PAD token: <|endoftext|>
Chat template available: True


In [12]:
import json
import numpy as np
from tqdm.auto import tqdm


def get_training_token_length(row):
    target = json.dumps(
        {
            "category": row["category"],
            "intent": row["intent"],
        },
        ensure_ascii=False,
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": str(row["instruction"]).strip(),
        },
        {
            "role": "assistant",
            "content": target,
        },
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
    )

    # Extract actual token IDs instead of counting dictionary keys.
    input_ids = encoded["input_ids"]

    # Handle both list and tensor-like outputs.
    if hasattr(input_ids, "tolist"):
        input_ids = input_ids.tolist()

    # Remove the batch dimension if present.
    if input_ids and isinstance(input_ids[0], list):
        input_ids = input_ids[0]

    return len(input_ids)


train_lengths = [
    get_training_token_length(row)
    for _, row in tqdm(
        train_df.iterrows(),
        total=len(train_df),
        desc="Training",
    )
]

val_lengths = [
    get_training_token_length(row)
    for _, row in tqdm(
        val_df.iterrows(),
        total=len(val_df),
        desc="Validation",
    )
]


def display_length_statistics(name, lengths):
    lengths = np.array(lengths)

    print(f"\n{'=' * 45}")
    print(name)
    print(f"{'=' * 45}")

    print("Examples:", len(lengths))
    print("Minimum:", lengths.min())
    print("Median:", np.percentile(lengths, 50))
    print("90th percentile:", np.percentile(lengths, 90))
    print("95th percentile:", np.percentile(lengths, 95))
    print("99th percentile:", np.percentile(lengths, 99))
    print("Maximum:", lengths.max())

    for limit in [64, 96, 128, 160, 192, 256]:
        count = np.sum(lengths > limit)
        percentage = count / len(lengths) * 100

        print(
            f"Longer than {limit:3d} tokens: "
            f"{count:5d} ({percentage:.2f}%)"
        )


display_length_statistics("TRAINING", train_lengths)
display_length_statistics("VALIDATION", val_lengths)

Training:   0%|          | 0/19419 [00:00<?, ?it/s]

Validation:   0%|          | 0/2427 [00:00<?, ?it/s]


TRAINING
Examples: 19419
Minimum: 49
Median: 59.0
90th percentile: 63.0
95th percentile: 65.0
99th percentile: 68.0
Maximum: 74
Longer than  64 tokens:   984 (5.07%)
Longer than  96 tokens:     0 (0.00%)
Longer than 128 tokens:     0 (0.00%)
Longer than 160 tokens:     0 (0.00%)
Longer than 192 tokens:     0 (0.00%)
Longer than 256 tokens:     0 (0.00%)

VALIDATION
Examples: 2427
Minimum: 50
Median: 59.0
90th percentile: 63.0
95th percentile: 65.0
99th percentile: 68.0
Maximum: 71
Longer than  64 tokens:   125 (5.15%)
Longer than  96 tokens:     0 (0.00%)
Longer than 128 tokens:     0 (0.00%)
Longer than 160 tokens:     0 (0.00%)
Longer than 192 tokens:     0 (0.00%)
Longer than 256 tokens:     0 (0.00%)


In [13]:
sample_row = train_df.iloc[0]

sample_length = get_training_token_length(sample_row)

print("Instruction:", sample_row["instruction"])
print("Category:", sample_row["category"])
print("Intent:", sample_row["intent"])
print("Actual token count:", sample_length)

sample_target = json.dumps(
    {
        "category": sample_row["category"],
        "intent": sample_row["intent"],
    },
    ensure_ascii=False,
)

sample_messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": str(sample_row["instruction"]).strip()},
    {"role": "assistant", "content": sample_target},
]

print("\nFormatted conversation:\n")

print(
    tokenizer.apply_chat_template(
        sample_messages,
        tokenize=False,
        add_generation_prompt=False,
    )
)

Instruction: I have got to change to the standard account, I need help
Category: ACCOUNT
Intent: switch_account
Actual token count: 61

Formatted conversation:

<|im_start|>system
Classify customer support requests. Return only a JSON object with keys "category" and "intent".<|im_end|>
<|im_start|>user
I have got to change to the standard account, I need help<|im_end|>
<|im_start|>assistant
{"category": "ACCOUNT", "intent": "switch_account"}<|im_end|>



**Preprocessing Pipeline**




In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

MAX_SEQ_LENGTH = 96

tokenizer.padding_side = "right"

PAD_TOKEN_ID = tokenizer.pad_token_id
EOS_TOKEN_ID = tokenizer.eos_token_id

print("Padding token ID:", PAD_TOKEN_ID)
print("EOS token ID:", EOS_TOKEN_ID)


class SupportDataset(Dataset):
    """
    Converts customer-support examples into tokenized conversations.

    Model input:
        System + user + assistant JSON + template trailing tokens

    Training labels:
        System and user tokens: -100
        Assistant JSON and EOS: actual token IDs
        Tokens after EOS: -100
    """

    def __init__(self, dataframe, tokenizer, max_length=96):
        self.examples = []

        for _, row in tqdm(
            dataframe.iterrows(),
            total=len(dataframe),
            desc="Preparing dataset",
        ):
            messages = build_messages_for_row(row)

            # Tokenize the complete conversation.
            full_encoding = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=False,
                return_dict=True,
            )

            # Tokenize the prompt without the assistant answer.
            prompt_encoding = tokenizer.apply_chat_template(
                messages[:2],
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
            )

            input_ids = full_encoding["input_ids"]
            prompt_ids = prompt_encoding["input_ids"]

            if hasattr(input_ids, "tolist"):
                input_ids = input_ids.tolist()

            if hasattr(prompt_ids, "tolist"):
                prompt_ids = prompt_ids.tolist()

            if input_ids and isinstance(input_ids[0], list):
                input_ids = input_ids[0]

            if prompt_ids and isinstance(prompt_ids[0], list):
                prompt_ids = prompt_ids[0]

            # Confirm prompt tokens exactly match the full conversation prefix.
            assert input_ids[:len(prompt_ids)] == prompt_ids

            if len(input_ids) > max_length:
                raise ValueError(
                    f"Example exceeds maximum length: {len(input_ids)}"
                )

            # Find the EOS token after the assistant prompt.
            eos_positions = [
                i
                for i in range(len(prompt_ids), len(input_ids))
                if input_ids[i] == EOS_TOKEN_ID
            ]

            if not eos_positions:
                raise ValueError(
                    "EOS token missing from assistant response."
                )

            eos_index = eos_positions[0]

            # Verify that only the expected template newline follows EOS.
            trailing_ids = input_ids[eos_index + 1:]
            trailing_text = tokenizer.decode(
                trailing_ids,
                skip_special_tokens=False,
            )

            if trailing_text != "\n":
                raise ValueError(
                    f"Unexpected tokens after EOS: {trailing_text!r}"
                )

            # Mask system and user tokens.
            # Train on assistant output through EOS.
            # Ignore the trailing template newline.
            labels = (
                [-100] * len(prompt_ids)
                + input_ids[len(prompt_ids):eos_index + 1]
                + [-100] * len(trailing_ids)
            )

            assert len(labels) == len(input_ids)
            assert labels[eos_index] == EOS_TOKEN_ID
            assert all(
                label == -100
                for label in labels[eos_index + 1:]
            )

            self.examples.append(
                {
                    "input_ids": input_ids,
                    "attention_mask": [1] * len(input_ids),
                    "labels": labels,
                }
            )

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


# Build both datasets.
train_dataset = SupportDataset(
    train_df,
    tokenizer,
    max_length=MAX_SEQ_LENGTH,
)

val_dataset = SupportDataset(
    val_df,
    tokenizer,
    max_length=MAX_SEQ_LENGTH,
)

print("\nDataset preparation complete.")
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

Padding token ID: 151643
EOS token ID: 151645


Preparing dataset:   0%|          | 0/19419 [00:00<?, ?it/s]

Preparing dataset:   0%|          | 0/2427 [00:00<?, ?it/s]


Dataset preparation complete.
Training examples: 19419
Validation examples: 2427


**Dynamic padding collator**

In [18]:
def support_data_collator(features):
    """
    Dynamically pads a batch of examples.

    input_ids: padded using the tokenizer's PAD token.
    attention_mask: 0 for padding, 1 for real tokens.
    labels: -100 for padding and ignored prompt tokens.
    """

    batch_max_length = max(
        len(feature["input_ids"])
        for feature in features
    )

    input_ids = []
    attention_masks = []
    labels = []

    for feature in features:
        padding_length = (
            batch_max_length - len(feature["input_ids"])
        )

        input_ids.append(
            feature["input_ids"]
            + [PAD_TOKEN_ID] * padding_length
        )

        attention_masks.append(
            feature["attention_mask"]
            + [0] * padding_length
        )

        labels.append(
            feature["labels"]
            + [-100] * padding_length
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(
            attention_masks,
            dtype=torch.long,
        ),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

**Create DataLoaders and inspect a batch**

In [33]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=support_data_collator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=support_data_collator,
)

print("Updated DataLoaders")
print("-------------------")
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Microbatch size:", BATCH_SIZE)
print("Accumulation steps:", GRADIENT_ACCUMULATION_STEPS)
print("Effective batch size:", BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Gradient checkpointing:", model.is_gradient_checkpointing)

Updated DataLoaders
-------------------
Training examples: 19419
Validation examples: 2427
Training batches: 2428
Validation batches: 304
Microbatch size: 8
Accumulation steps: 2
Effective batch size: 16
Gradient checkpointing: False


**Load basemodel in 4bit quantization**

In [20]:
import torch
import bitsandbytes as bnb

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
)

#check GPU availability

assert torch.cuda.is_available(), "CUDA GPU is not available."

gpu_name = torch.cuda.get_device_name(0)

print("GPU:", gpu_name)
print("CUDA:", torch.version.cuda)

assert "T4" in gpu_name, (
    f"Expected Tesla T4, but found {gpu_name}"
)


#configuration 4bit quantization, double quantization

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)


#load the model - 4bit quantization

print("\nLoading Qwen in 4-bit mode...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model.config.use_cache = False

print("\nModel loaded successfully.")

print("Model class:", model.__class__.__name__)
print("Model device:", model.device)
print("Model parameter dtype:", model.dtype)
print("Quantization:", model.config.quantization_config)


# Inspect - GPU memory
allocated_memory = torch.cuda.memory_allocated(0) / (1024 ** 3)
reserved_memory = torch.cuda.memory_reserved(0) / (1024 ** 3)

print("\nGPU memory usage:")
print(f"Allocated: {allocated_memory:.2f} GB")
print(f"Reserved: {reserved_memory:.2f} GB")


#model architexture

print("\nModel architecture:")
print(model.config.architectures)

print("\nLinear layer names:")

linear_layer_names = set()

for name, module in model.named_modules():
    if isinstance(module, bnb.nn.Linear4bit):
        linear_layer_names.add(name.split(".")[-1])

print(sorted(linear_layer_names))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


GPU: Tesla T4
CUDA: 13.0

Loading Qwen in 4-bit mode...


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded successfully.
Model class: Qwen2ForCausalLM
Model device: cuda:0
Model parameter dtype: torch.float16
Quantization: BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}


GPU memory usage:
Allocated: 0.43 GB
Reserved: 0.45 GB

Model architecture:
['Qwen2ForCausalLM']

Linear layer names:
['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']


In [22]:
from peft import prepare_model_for_kbit_training

# Prepare the quantized model for adapter training.
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },
)

print("Model prepared for k-bit training.")

print("Gradient checkpointing enabled:",
      model.is_gradient_checkpointing)

print("Model cache enabled:", model.config.use_cache)

Model prepared for k-bit training.
Gradient checkpointing enabled: True
Model cache enabled: False


In [23]:
from peft import LoraConfig, get_peft_model

#lora configuration

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)



model = get_peft_model(
    model,
    lora_config,
)


# Inspect trainable parameters


model.print_trainable_parameters()

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_percentage = (
    trainable_parameters / total_parameters
) * 100

print("\nParameter verification")
print("----------------------")
print(f"Total parameters: {total_parameters:,}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Frozen parameters: {total_parameters - trainable_parameters:,}")
print(f"Trainable percentage: {trainable_percentage:.4f}%")


# Verify adapter configuration


print("\nLoRA configuration:")
print("Rank:", lora_config.r)
print("Alpha:", lora_config.lora_alpha)
print("Dropout:", lora_config.lora_dropout)
print("Target modules:", lora_config.target_modules)

assert trainable_parameters > 0
assert trainable_parameters < total_parameters

assert model.config.use_cache is False

print("\nLoRA adapter verification successful.")

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497

Parameter verification
----------------------
Total parameters: 323,917,696
Trainable parameters: 8,798,208
Frozen parameters: 315,119,488
Trainable percentage: 2.7162%

LoRA configuration:
Rank: 16
Alpha: 32
Dropout: 0.05
Target modules: {'q_proj', 'v_proj', 'o_proj', 'gate_proj', 'down_proj', 'k_proj', 'up_proj'}

LoRA adapter verification successful.


In [24]:
print("First 20 trainable parameter names:\n")

trainable_names = []

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        trainable_names.append(name)

for name in trainable_names[:20]:
    print(name)

assert all(
    "lora_" in name
    for name in trainable_names
)

print("\nTotal trainable tensors:", len(trainable_names))
print("All trainable parameters belong to LoRA adapters.")

First 20 trainable parameter names:

base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight
base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.up_proj.lora_B.default.weight
base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight
base_model.model.model.layers.0.mlp.down_proj.lora_B.defaul

**OPtimizer and Scheduler**

In [35]:
import math
from transformers import get_linear_schedule_with_warmup

EPOCHS = 3
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0

# Preserve the existing AdamW optimizer and its state.
for param_group in optimizer.param_groups:
    param_group["lr"] = LEARNING_RATE

updates_per_epoch = math.ceil(
    len(train_loader) / GRADIENT_ACCUMULATION_STEPS
)

total_training_steps = updates_per_epoch * EPOCHS

warmup_steps = math.ceil(
    total_training_steps * WARMUP_RATIO
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

print("Updated training schedule")
print("-------------------------")
print("Preserved model optimizer updates:", global_step)
print("Updates per epoch:", updates_per_epoch)
print("Additional planned updates:", total_training_steps)
print("Warmup updates:", warmup_steps)
print("Current learning rate:", scheduler.get_last_lr()[0])

assert global_step == 100
assert model.is_gradient_checkpointing is False
assert len(optimizer.param_groups[0]["params"]) == 336

print("\nConfiguration verification successful.")

Updated training schedule
-------------------------
Preserved model optimizer updates: 100
Updates per epoch: 1214
Additional planned updates: 3642
Warmup updates: 110
Current learning rate: 0.0

Configuration verification successful.


**Training**

In [36]:
import os
import shutil
import torch

from pathlib import Path
from tqdm.auto import tqdm


OUTPUT_DIR = Path("/content/outputs")
LATEST_CHECKPOINT_DIR = OUTPUT_DIR / "latest_checkpoint"
BEST_ADAPTER_DIR = OUTPUT_DIR / "best_adapter"

CHECKPOINT_INTERVAL = 200

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def get_model_device(model):
    """Get the device used by the model."""
    return next(model.parameters()).device


def count_supervised_tokens(labels):
    """Count tokens that contribute to the training loss."""
    return (labels != -100).sum().item()


def validate_model(model, dataloader):
    """
    Calculate token-weighted validation loss.

    Prompt tokens and padding tokens are excluded because
    their labels are set to -100.
    """
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    device = get_model_device(model)

    with torch.inference_mode():

        for batch in tqdm(dataloader, desc="Validation", leave=False):

            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            supervised_tokens = count_supervised_tokens(
                batch["labels"]
            )

            outputs = model(**batch)

            loss = outputs.loss

            if not torch.isfinite(loss):
                raise RuntimeError("Non-finite validation loss detected.")

            total_loss += loss.item() * supervised_tokens
            total_tokens += supervised_tokens

    if total_tokens == 0:
        raise RuntimeError("No supervised validation tokens found.")

    average_loss = total_loss / total_tokens

    model.train()

    return average_loss


def save_training_checkpoint(
    model,
    tokenizer,
    optimizer,
    scheduler,
    global_step,
    epoch_index,
    batches_consumed,
    epoch_loss_sum,
    epoch_token_count,
    best_validation_loss,
    history,
):
    """
    Save the LoRA adapter and all information needed
    to restore training progress.
    """

    temporary_dir = OUTPUT_DIR / "latest_checkpoint_tmp"
    backup_dir = OUTPUT_DIR / "latest_checkpoint_backup"

    # Remove an incomplete temporary checkpoint, if present.
    if temporary_dir.exists():
        shutil.rmtree(temporary_dir)

    temporary_dir.mkdir(parents=True)

    adapter_dir = temporary_dir / "adapter"
    adapter_dir.mkdir()

    # Save only the trainable LoRA adapter, not the full base model.
    model.save_pretrained(
        str(adapter_dir),
        safe_serialization=True,
    )

    tokenizer.save_pretrained(str(adapter_dir))

    # Save optimizer, scheduler, and training progress.
    training_state = {
        "global_step": global_step,
        "epoch_index": epoch_index,
        "batches_consumed": batches_consumed,
        "epoch_loss_sum": epoch_loss_sum,
        "epoch_token_count": epoch_token_count,
        "best_validation_loss": best_validation_loss,
        "history": history,
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
    }

    torch.save(
        training_state,
        temporary_dir / "training_state.pt",
    )

    # Replace the previous checkpoint only after the new one is saved.
    if backup_dir.exists():
        shutil.rmtree(backup_dir)

    if LATEST_CHECKPOINT_DIR.exists():
        os.replace(LATEST_CHECKPOINT_DIR, backup_dir)

    os.replace(temporary_dir, LATEST_CHECKPOINT_DIR)

    if backup_dir.exists():
        shutil.rmtree(backup_dir)

    print(f"\nCheckpoint saved at optimizer update {global_step}.")

In [37]:
# We are continuing from the 100 optimizer updates
# already completed in the previous training attempt.

start_epoch = 0
batches_consumed = 0

epoch_loss_sum = 0.0
epoch_token_count = 0

best_validation_loss = float("inf")
history = []

TRAINING_SEED = 42

optimizer.zero_grad(set_to_none=True)

print("Training state initialized")
print("---------------------------")
print("Existing optimizer updates:", global_step)
print("Starting epoch:", start_epoch + 1)
print("Batches already consumed:", batches_consumed)
print("Best validation loss:", best_validation_loss)
print("Checkpoint interval:", CHECKPOINT_INTERVAL)

assert global_step == 100
assert len(train_loader) == 2428
assert len(val_loader) == 304

print("\nTraining state verification successful.")

Training state initialized
---------------------------
Existing optimizer updates: 100
Starting epoch: 1
Batches already consumed: 0
Best validation loss: inf
Checkpoint interval: 200

Training state verification successful.


In [38]:
import math
from itertools import islice
from torch.utils.data import DataLoader


def train_model():

    global global_step
    global start_epoch
    global batches_consumed
    global epoch_loss_sum
    global epoch_token_count
    global best_validation_loss
    global history

    device = get_model_device(model)

    model.train()

    for epoch_index in range(start_epoch, EPOCHS):

        # Create a reproducible shuffled DataLoader for this epoch.
        # This allows us to recreate the same batch order when resuming.
        generator = torch.Generator()
        generator.manual_seed(TRAINING_SEED + epoch_index)

        epoch_train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            generator=generator,
            collate_fn=support_data_collator,
        )

        iterator = iter(epoch_train_loader)

        # Skip batches already processed if resuming an interrupted epoch.
        for _ in range(batches_consumed):
            try:
                next(iterator)
            except StopIteration:
                raise RuntimeError(
                    "Checkpoint batch position exceeds DataLoader length."
                )

        progress = tqdm(
            total=len(epoch_train_loader),
            initial=batches_consumed,
            desc=f"Epoch {epoch_index + 1}/{EPOCHS}",
        )

        while True:

            # Collect one gradient-accumulation group.
            group = list(
                islice(iterator, GRADIENT_ACCUMULATION_STEPS)
            )

            if not group:
                break

            group_token_count = sum(
                count_supervised_tokens(batch["labels"])
                for batch in group
            )

            if group_token_count == 0:
                raise RuntimeError(
                    "Accumulation group contains no supervised tokens."
                )

            optimizer.zero_grad(set_to_none=True)

            # Process each microbatch in the accumulation group.
            for batch in group:

                batch = {
                    key: value.to(device)
                    for key, value in batch.items()
                }

                supervised_tokens = count_supervised_tokens(
                    batch["labels"]
                )

                outputs = model(**batch)

                loss = outputs.loss

                if not torch.isfinite(loss):
                    raise RuntimeError(
                        f"Non-finite training loss at step {global_step}."
                    )

                # Weight each microbatch loss by its share of
                # supervised tokens in this accumulation group.
                scaled_loss = (
                    loss * supervised_tokens / group_token_count
                )

                scaled_loss.backward()

                epoch_loss_sum += (
                    loss.item() * supervised_tokens
                )

                epoch_token_count += supervised_tokens

            # Clip gradients to prevent excessively large updates.
            trainable_parameters = [
                parameter
                for parameter in model.parameters()
                if parameter.requires_grad
            ]

            gradient_norm = torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                MAX_GRAD_NORM,
            )

            optimizer.step()
            scheduler.step()

            global_step += 1
            batches_consumed += len(group)

            current_lr = scheduler.get_last_lr()[0]

            progress.update(len(group))
            progress.set_postfix(
                {
                    "step": global_step,
                    "loss": f"{epoch_loss_sum / epoch_token_count:.4f}",
                    "lr": f"{current_lr:.2e}",
                    "grad": f"{gradient_norm.item():.3f}",
                }
            )

            # Save a recovery checkpoint periodically.
            if global_step % CHECKPOINT_INTERVAL == 0:

                save_training_checkpoint(
                    model=model,
                    tokenizer=tokenizer,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    global_step=global_step,
                    epoch_index=epoch_index,
                    batches_consumed=batches_consumed,
                    epoch_loss_sum=epoch_loss_sum,
                    epoch_token_count=epoch_token_count,
                    best_validation_loss=best_validation_loss,
                    history=history,
                )

        progress.close()

        # Calculate the average training loss for this epoch.
        train_loss = epoch_loss_sum / epoch_token_count

        # Evaluate the model on the validation dataset.
        validation_loss = validate_model(
            model,
            val_loader,
        )

        epoch_record = {
            "epoch": epoch_index + 1,
            "global_step": global_step,
            "train_loss": train_loss,
            "validation_loss": validation_loss,
            "learning_rate": scheduler.get_last_lr()[0],
        }

        history.append(epoch_record)

        print(f"\nEpoch {epoch_index + 1} completed")
        print("--------------------------------")
        print(f"Training loss:   {train_loss:.4f}")
        print(f"Validation loss: {validation_loss:.4f}")
        print(f"Global step:     {global_step}")

        # Save the adapter whenever validation loss improves.
        if validation_loss < best_validation_loss:

            best_validation_loss = validation_loss

            if BEST_ADAPTER_DIR.exists():
                shutil.rmtree(BEST_ADAPTER_DIR)

            model.save_pretrained(
                str(BEST_ADAPTER_DIR),
                safe_serialization=True,
            )

            tokenizer.save_pretrained(str(BEST_ADAPTER_DIR))

            print("New best adapter saved.")

        # Prepare progress information for the next epoch.
        start_epoch = epoch_index + 1
        batches_consumed = 0
        epoch_loss_sum = 0.0
        epoch_token_count = 0

        # Save at every epoch boundary as well.
        save_training_checkpoint(
            model=model,
            tokenizer=tokenizer,
            optimizer=optimizer,
            scheduler=scheduler,
            global_step=global_step,
            epoch_index=start_epoch,
            batches_consumed=0,
            epoch_loss_sum=0.0,
            epoch_token_count=0,
            best_validation_loss=best_validation_loss,
            history=history,
        )

        # Save training history.
        import json

        with open(
            OUTPUT_DIR / "training_history.json",
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(history, file, indent=4)

    print("\nTraining completed.")
    print("-------------------")
    print("Total optimizer updates:", global_step)
    print("Best validation loss:", best_validation_loss)
    print("Best adapter:", BEST_ADAPTER_DIR)
    print("Latest checkpoint:", LATEST_CHECKPOINT_DIR)


train_model()

Epoch 1/3:   0%|          | 0/2428 [00:00<?, ?it/s]


Checkpoint saved at optimizer update 200.

Checkpoint saved at optimizer update 400.

Checkpoint saved at optimizer update 600.

Checkpoint saved at optimizer update 800.

Checkpoint saved at optimizer update 1000.

Checkpoint saved at optimizer update 1200.


Validation:   0%|          | 0/304 [00:00<?, ?it/s]


Epoch 1 completed
--------------------------------
Training loss:   0.0030
Validation loss: 0.0005
Global step:     1314
New best adapter saved.

Checkpoint saved at optimizer update 1314.


Epoch 2/3:   0%|          | 0/2428 [00:00<?, ?it/s]


Checkpoint saved at optimizer update 1400.

Checkpoint saved at optimizer update 1600.

Checkpoint saved at optimizer update 1800.

Checkpoint saved at optimizer update 2000.

Checkpoint saved at optimizer update 2200.

Checkpoint saved at optimizer update 2400.


... (2 outputs left out)

Epoch 3/3:   0%|          | 0/2428 [00:00<?, ?it/s]


Checkpoint saved at optimizer update 2600.

Checkpoint saved at optimizer update 2800.

Checkpoint saved at optimizer update 3000.

Checkpoint saved at optimizer update 3200.

Checkpoint saved at optimizer update 3400.

Checkpoint saved at optimizer update 3600.


Validation:   0%|          | 0/304 [00:00<?, ?it/s]


Epoch 3 completed
--------------------------------
Training loss:   0.0001
Validation loss: 0.0006
Global step:     3742

Checkpoint saved at optimizer update 3742.

Training completed.
-------------------
Total optimizer updates: 3742
Best validation loss: 0.0004628384928379882
Best adapter: /content/outputs/best_adapter
Latest checkpoint: /content/outputs/latest_checkpoint


In [39]:
import json
from pathlib import Path

OUTPUT_DIR = Path("/content/outputs")
BEST_ADAPTER_DIR = OUTPUT_DIR / "best_adapter"
LATEST_CHECKPOINT_DIR = OUTPUT_DIR / "latest_checkpoint"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"

print("BEST ADAPTER FILES")
print("------------------")

for path in sorted(BEST_ADAPTER_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(BEST_ADAPTER_DIR)}: {path.stat().st_size:,} bytes")

print("\nLATEST CHECKPOINT FILES")
print("-----------------------")

for path in sorted(LATEST_CHECKPOINT_DIR.rglob("*")):
    if path.is_file():
        print(f"{path.relative_to(LATEST_CHECKPOINT_DIR)}: {path.stat().st_size:,} bytes")

print("\nTRAINING HISTORY")
print("----------------")

with open(HISTORY_PATH, "r", encoding="utf-8") as file:
    training_history = json.load(file)

for record in training_history:
    print(record)

assert (BEST_ADAPTER_DIR / "adapter_config.json").exists()
assert (
    (BEST_ADAPTER_DIR / "adapter_model.safetensors").exists()
    or (BEST_ADAPTER_DIR / "adapter_model.bin").exists()
)
assert (LATEST_CHECKPOINT_DIR / "training_state.pt").exists()
assert len(training_history) == 3

print("\nArtifact verification successful.")

BEST ADAPTER FILES
------------------
README.md: 5,206 bytes
adapter_config.json: 1,182 bytes
adapter_model.safetensors: 35,237,104 bytes
chat_template.jinja: 2,507 bytes
tokenizer.json: 11,421,892 bytes
tokenizer_config.json: 694 bytes

LATEST CHECKPOINT FILES
-----------------------
adapter/README.md: 5,206 bytes
adapter/adapter_config.json: 1,182 bytes
adapter/adapter_model.safetensors: 35,237,104 bytes
adapter/chat_template.jinja: 2,507 bytes
adapter/tokenizer.json: 11,421,892 bytes
adapter/tokenizer_config.json: 694 bytes
training_state.pt: 70,673,825 bytes

TRAINING HISTORY
----------------
{'epoch': 1, 'global_step': 1314, 'train_loss': 0.003010327331935121, 'validation_loss': 0.00047105184309754973, 'learning_rate': 0.0001374858437146093}
{'epoch': 2, 'global_step': 2528, 'train_loss': 0.00033165918055744973, 'validation_loss': 0.0004628384928379882, 'learning_rate': 6.874292185730464e-05}
{'epoch': 3, 'global_step': 3742, 'train_loss': 8.069595155634435e-05, 'validation_loss':

# **Final Evaluation of the fine-tuned model against the baseline**

In [44]:
from pathlib import Path

BEST_ADAPTER_DIR = Path("/content/outputs/best_adapter")

assert BEST_ADAPTER_DIR.exists(), "Best adapter directory not found."

required_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
]

for filename in required_files:
    file_path = BEST_ADAPTER_DIR / filename
    assert file_path.exists(), f"Missing: {filename}"
    print(f"Verified: {filename}")

print("\nBest adapter is ready.")

Verified: adapter_config.json
Verified: adapter_model.safetensors
Verified: tokenizer.json
Verified: tokenizer_config.json

Best adapter is ready.


**Verify the test data**

In [50]:
import pandas as pd

test_df = pd.read_csv(TEST_PATH)

print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

print("\nMissing values:")
print(test_df[["instruction", "category", "intent"]].isna().sum())

assert len(test_df) == 2428
assert test_df[["instruction", "category", "intent"]].notna().all().all()

print("\nTest dataset verification successful.")

Shape: (2428, 6)
Columns: ['flags', 'instruction', 'category', 'intent', 'response', 'normalized_instruction']

Missing values:
instruction    0
category       0
intent         0
dtype: int64

Test dataset verification successful.


**Load the Fine tuned Model**

In [53]:
import torch
from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
BEST_ADAPTER_DIR = "/content/outputs/best_adapter"

assert torch.cuda.is_available(), "CUDA GPU is required."

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
    low_cpu_mem_usage=True,
)

model = PeftModel.from_pretrained(
    base_model,
    BEST_ADAPTER_DIR,
)

model.eval()

print("Model loaded successfully.")
print("Model class:", model.__class__.__name__)
print("Device:", model.get_input_embeddings().weight.device)
print("GPU:", torch.cuda.get_device_name(0))
print("Gradient checkpointing:", model.is_gradient_checkpointing)
print("Training mode:", model.training)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully.
Model class: PeftModelForCausalLM
Device: cuda:0
GPU: Tesla T4
Gradient checkpointing: False
Training mode: False


In [54]:
import torch

SYSTEM_PROMPT = (
    "Classify customer support requests. "
    'Return only a JSON object with keys "category" and "intent".'
)

test_instruction = (
    "I placed an order last week and want to know where it is."
)

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_instruction},
]

# Apply the model's native Qwen chat template.
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

# Move input tensors to the GPU.
inputs = {
    key: value.to(model.get_input_embeddings().weight.device)
    for key, value in inputs.items()
}

# Generate a prediction.
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=48,
        do_sample=False,
        num_beams=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

# Decode only newly generated tokens.
generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True,
)

print("INPUT:")
print(test_instruction)

print("\nMODEL OUTPUT:")
print(prediction)

INPUT:
I placed an order last week and want to know where it is.

MODEL OUTPUT:
{"category": "ORDER", "intent": "track_order"}


In [55]:
import torch
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

BATCH_SIZE = 16
MAX_NEW_TOKENS = 48
MAX_INPUT_LENGTH = 96

OUTPUT_DIR = Path("/content/outputs/fine_tuned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = OUTPUT_DIR / "fine_tuned_predictions.csv"

# Left padding is recommended for batched decoder-only generation.
tokenizer.padding_side = "left"

# Enable KV caching for inference.
model.config.use_cache = True

#prepare inference prompt - apply chat template of the tokenizer

def build_inference_prompt(instruction):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": str(instruction).strip()},
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


prompts = [
    build_inference_prompt(instruction)
    for instruction in test_df["instruction"]
]

print("Total test examples:", len(prompts))
print("First prompt preview:")
print(prompts[0])

#generate predictions

all_predictions = []

device = model.get_input_embeddings().weight.device

model.eval()

for start in tqdm(
    range(0, len(prompts), BATCH_SIZE),
    desc="Fine-tuned inference",
):
    batch_prompts = prompts[start:start + BATCH_SIZE]

    # Tokenize the batch.
    inputs = tokenizer(
        batch_prompts,
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        add_special_tokens=False,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate using deterministic greedy decoding.
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Remove the input prompt tokens.
    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]

    # Decode only the generated assistant response.
    batch_outputs = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True,
    )

    all_predictions.extend(
        output.strip() for output in batch_outputs
    )

#prediction count

assert len(all_predictions) == len(test_df)

print("\nInference completed.")
print("Predictions generated:", len(all_predictions))
print("Empty predictions:", sum(not x for x in all_predictions))



prediction_df = test_df[
    ["instruction", "category", "intent"]
].copy()

prediction_df["raw_output"] = all_predictions

prediction_df.to_csv(
    PREDICTIONS_PATH,
    index=False,
    encoding="utf-8",
)

print("\nSaved predictions:", PREDICTIONS_PATH)
print("Output shape:", prediction_df.shape)

print("\nSample predictions:")
display(prediction_df.head(10))

print("\nFile size:",
      f"{PREDICTIONS_PATH.stat().st_size / (1024**2):.2f} MB")

Total test examples: 2428
First prompt preview:
<|im_start|>system
Classify customer support requests. Return only a JSON object with keys "category" and "intent".<|im_end|>
<|im_start|>user
help me to see how long it takes for the article to arrive<|im_end|>
<|im_start|>assistant



Fine-tuned inference:   0%|          | 0/152 [00:00<?, ?it/s]


Inference completed.
Predictions generated: 2428
Empty predictions: 0

Saved predictions: /content/outputs/fine_tuned/fine_tuned_predictions.csv
Output shape: (2428, 4)

Sample predictions:


,instruction,category,intent,raw_output
0,help me to see how long it takes for the artic...,DELIVERY,delivery_period,"{""category"": ""DELIVERY"", ""intent"": ""delivery_p..."
1,can you give me information about opening {{Ac...,ACCOUNT,create_account,"{""category"": ""ACCOUNT"", ""intent"": ""create_acco..."
2,i have got to buy some items how can i do it,ORDER,place_order,"{""category"": ""ORDER"", ""intent"": ""place_order""}"
3,I can't notify of problems with a registration,ACCOUNT,registration_problems,"{""category"": ""ACCOUNT"", ""intent"": ""registratio..."
4,I don't know what I have to do to create a {{A...,ACCOUNT,create_account,"{""category"": ""ACCOUNT"", ""intent"": ""create_acco..."
5,where do i check the cancellation penalty,CANCEL,check_cancellation_fee,"{""category"": ""CANCEL"", ""intent"": ""check_cancel..."
6,assistance to check hw soon can i expect the o...,DELIVERY,delivery_period,"{""category"": ""DELIVERY"", ""intent"": ""delivery_p..."
7,I need help to talk to a person,CONTACT,contact_human_agent,"{""category"": ""CONTACT"", ""intent"": ""contact_hum..."
8,I don't know how to correct the data included ...,ACCOUNT,edit_account,"{""category"": ""ACCOUNT"", ""intent"": ""edit_account""}"
9,want help to cancel order {{Order Number}},ORDER,cancel_order,"{""category"": ""ORDER"", ""intent"": ""cancel_order""}"



File size: 0.32 MB


In [56]:
import json
import pandas as pd

def parse_output(value):
    try:
        result = json.loads(str(value).strip())
        return result if isinstance(result, dict) else None
    except (json.JSONDecodeError, TypeError):
        return None


parsed = prediction_df["raw_output"].apply(parse_output)

valid_json_count = parsed.notna().sum()

schema_valid_count = sum(
    isinstance(item, dict)
    and isinstance(item.get("category"), str)
    and bool(item["category"].strip())
    and isinstance(item.get("intent"), str)
    and bool(item["intent"].strip())
    for item in parsed
)

valid_categories = set(test_df["category"].str.strip().str.upper())
valid_intents = set(test_df["intent"].str.strip().str.upper())

category_valid_count = sum(
    isinstance(item, dict)
    and isinstance(item.get("category"), str)
    and item["category"].strip().upper() in valid_categories
    for item in parsed
)

intent_valid_count = sum(
    isinstance(item, dict)
    and isinstance(item.get("intent"), str)
    and item["intent"].strip().upper() in valid_intents
    for item in parsed
)

print("Total predictions:", len(prediction_df))
print("Empty outputs:", prediction_df["raw_output"].str.strip().eq("").sum())
print("Valid JSON:", valid_json_count)
print("Valid JSON schema:", schema_valid_count)
print("Valid category labels:", category_valid_count)
print("Valid intent labels:", intent_valid_count)

print("\nInspect row 9:")
print("Instruction:", repr(prediction_df.iloc[9]["instruction"]))
print("True labels:", prediction_df.iloc[9]["category"], prediction_df.iloc[9]["intent"])
print("Raw output:", repr(prediction_df.iloc[9]["raw_output"]))

print("\nFirst five raw outputs:")
for i, output in enumerate(prediction_df["raw_output"].head(5)):
    print(i, repr(output))

Total predictions: 2428
Empty outputs: 0
Valid JSON: 2428
Valid JSON schema: 2428
Valid category labels: 2428
Valid intent labels: 2428

Inspect row 9:
Instruction: 'want help to cancel order {{Order Number}}'
True labels: ORDER cancel_order
Raw output: '{"category": "ORDER", "intent": "cancel_order"}'

First five raw outputs:
0 '{"category": "DELIVERY", "intent": "delivery_period"}'
1 '{"category": "ACCOUNT", "intent": "create_account"}'
2 '{"category": "ORDER", "intent": "place_order"}'
3 '{"category": "ACCOUNT", "intent": "registration_problems"}'
4 '{"category": "ACCOUNT", "intent": "create_account"}'


In [57]:
from google.colab import files

files.download(
    "/content/outputs/fine_tuned/fine_tuned_predictions.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>